# Data.gov/USGS live earthquake feed with graceful fallback

This notebook demonstrates live open-data ingestion. It attempts to read the USGS GeoJSON earthquake feed and falls back to an embedded local concept table if the browser blocks network access.

Because the feed changes continuously, record the retrieval date when using it in a report.

**Reflection questions:** How does recency affect interpretation? What map symbol should represent magnitude versus depth? How would you avoid alarming users with incomplete real-time data?

In [ ]:
# Pyodide/JupyterLite bootstrap: install only pure-Python packages used in this notebook.
import sys, importlib
try:
    import micropip
except Exception:
    micropip = None

async def ensure_packages(packages):
    for pkg, import_name in packages:
        try:
            importlib.import_module(import_name)
        except Exception:
            if micropip is None:
                raise RuntimeError(f'{pkg} is not installed and micropip is unavailable.')
            await micropip.install(pkg)

await ensure_packages([('pandas','pandas'), ('folium','folium'), ('branca','branca'), ('plotly','plotly')])


In [ ]:
from pathlib import Path
import json, math, statistics
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap, TimestampedGeoJson, MiniMap, Fullscreen, MeasureControl

DATA = Path('../data')

def load_json(name):
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def load_csv(name):
    return pd.read_csv(DATA / name)

def add_standard_controls(m):
    MiniMap(toggle_display=True).add_to(m)
    Fullscreen().add_to(m)
    MeasureControl(primary_length_unit='kilometers').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

def color_scale(values, colors=('green','orange','red')):
    vals = list(values)
    lo, hi = min(vals), max(vals)
    def pick(v):
        if hi == lo:
            return colors[1]
        t = (v - lo) / (hi - lo)
        return colors[0] if t < .33 else colors[1] if t < .66 else colors[2]
    return pick


In [ ]:
import json, pandas as pd
from pyodide.http import pyfetch

async def get_usgs_feed():
    url = 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_day.geojson'
    try:
        response = await pyfetch(url)
        return await response.json()
    except Exception as e:
        print('Live fetch failed; using fallback examples:', e)
        return {'features':[
            {'properties':{'place':'Fallback: California coast','mag':3.1,'time':0}, 'geometry':{'coordinates':[-122.8,38.8,7]}},
            {'properties':{'place':'Fallback: Puerto Rico region','mag':3.6,'time':0}, 'geometry':{'coordinates':[-66.6,18.1,12]}},
            {'properties':{'place':'Fallback: Alaska peninsula','mag':4.2,'time':0}, 'geometry':{'coordinates':[-155.2,57.1,30]}},
        ]}

feed = await get_usgs_feed()
rows=[]
for f in feed['features'][:100]:
    lon, lat, depth = f['geometry']['coordinates']
    rows.append({'place':f['properties']['place'], 'mag':f['properties']['mag'], 'lat':lat, 'lon':lon, 'depth_km':depth})
quakes = pd.DataFrame(rows)
quakes.head()

In [ ]:
m = folium.Map(location=[20,-155], zoom_start=3, tiles='CartoDB positron')
for _, r in quakes.iterrows():
    folium.CircleMarker([r.lat,r.lon], radius=max(3, r.mag*2), fill=True, popup=f"{r.place}<br>M {r.mag}<br>Depth {r.depth_km} km").add_to(m)
add_standard_controls(m)
m